<a href="https://colab.research.google.com/github/fofugit/understat-player-explorer/blob/main/understat_player_shot_explorer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Understat Player Shot Explorer (Colab)

This notebook builds a simple interactive app for exploring **player-level shot data** from Understat in Google Colab.

## What it does
- choose a **league**
- choose a **season**
- choose a **team**
- type/select a **player**
- view the player's **full shot-level table**
- view **mean match xG**, **standard deviation**, **variance**, and a **histogram**
- download the filtered shot table as a **CSV**

## Important note on xA and related variables
Understat's `get_player_shots()` endpoint returns shot-level information, while `get_player_matches()` returns per-match player statistics such as `xA`, `assists`, `key_passes`, `xGChain`, and `xGBuildup`. In this notebook, those match-level fields are merged onto the shot table by `match_id` so they are available in the exported table too.

In [ ]:
# @title
from IPython.display import HTML

HTML("""
<script>
function toggleCode(){
  let codeCells = document.querySelectorAll('.code-cell');
  codeCells.forEach(cell => {
    cell.style.display = 'none';
  });
}
window.onload = toggleCode;
</script>
""")

In [ ]:
# @title

# Install packages
!pip -q install understat aiohttp nest_asyncio pandas numpy matplotlib ipywidgets aiohttp

In [ ]:
# @title
# =========================================================
# UNDERSTAT PLAYER EXPLORER
# Google Colab single-cell app
# =========================================================

import asyncio
import aiohttp
import nest_asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from understat import Understat

nest_asyncio.apply()

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    colab_files = None
    IN_COLAB = False

# -----------------------------
# PANDAS DISPLAY OPTIONS
# -----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", 200)

# -----------------------------
# CONFIG
# -----------------------------
LEAGUE_OPTIONS = [
    ("EPL", "epl"),
    ("La Liga", "la_liga"),
    ("Bundesliga", "bundesliga"),
    ("Serie A", "serie_a"),
    ("Ligue 1", "ligue_1"),
    ("RFPL", "rfpl"),
]

SEASON_START_YEARS = list(range(2014, 2026))
SEASON_OPTIONS = [("All seasons", "all")] + [(str(y), y) for y in SEASON_START_YEARS]

SHOT_NUMERIC_COLS = [
    "id", "minute", "X", "Y", "xG", "player_id", "match_id", "h_goals", "a_goals", "season"
]

MATCH_NUMERIC_COLS = [
    "id", "goals", "shots", "time", "h_goals", "a_goals", "xG", "xA", "assists",
    "key_passes", "npg", "npxG", "xGChain", "xGBuildup", "season"
]

# -----------------------------
# APP STATE
# -----------------------------
APP = {
    "roster_df": None,
    "current_table": None,
    "current_mode": None,
    "current_player_name": None,
    "current_league": None,
    "current_season": None,
    "current_team": None,
}

# -----------------------------
# ASYNC HELPERS
# -----------------------------
def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)

async def fetch_league_players_and_teams_one(league_code, season_start):
    async with aiohttp.ClientSession() as session:
        u = Understat(session)
        players = await u.get_league_players(league_code, season_start)
        teams = await u.get_teams(league_code, season_start)
    return players, teams

async def fetch_player_bundle(player_id):
    async with aiohttp.ClientSession() as session:
        u = Understat(session)
        shots = await u.get_player_shots(player_id)
        matches = await u.get_player_matches(player_id)
        try:
            grouped = await u.get_player_grouped_stats(player_id)
        except Exception:
            grouped = None
    return shots, matches, grouped

# -----------------------------
# CLEANERS
# -----------------------------
def clean_players_df(players_raw, league_code, season_start):
    df = pd.DataFrame(players_raw).copy()
    if df.empty:
        return df

    if "id" in df.columns:
        df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")

    df["selected_league"] = league_code
    df["selected_season"] = season_start

    if "team_title" in df.columns:
        df["team_title"] = df["team_title"].astype(str)

    keep_first = [
        "id", "player_name", "team_title", "position", "games", "time", "goals", "shots",
        "xG", "xA", "assists", "key_passes", "npg", "npxG", "xGChain", "xGBuildup",
        "selected_league", "selected_season"
    ]
    keep_first = [c for c in keep_first if c in df.columns]
    rest = [c for c in df.columns if c not in keep_first]
    df = df[keep_first + rest]

    sort_cols = [c for c in ["team_title", "player_name"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)

    return df

def clean_teams_df(teams_raw):
    rows = []

    if isinstance(teams_raw, dict):
        iterable = teams_raw.values()
    else:
        iterable = teams_raw

    for team in iterable:
        if isinstance(team, dict):
            rows.append({
                "team_id": team.get("id"),
                "team_title": team.get("title"),
                "team_short_title": team.get("short_title")
            })

    df = pd.DataFrame(rows).drop_duplicates()
    if not df.empty and "team_title" in df.columns:
        df = df[df["team_title"].notna()].copy()
        df["team_title"] = df["team_title"].astype(str)
        df = df.sort_values("team_title").reset_index(drop=True)
    return df

def clean_shots_df(shots_raw):
    df = pd.DataFrame(shots_raw).copy()
    if df.empty:
        return df

    rename_map = {
        "id": "shot_id",
        "h_team": "home_team",
        "a_team": "away_team",
    }
    df = df.rename(columns=rename_map)

    for col in SHOT_NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "player_id" in df.columns:
        df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")
    if "match_id" in df.columns:
        df["match_id"] = pd.to_numeric(df["match_id"], errors="coerce").astype("Int64")
    if "season" in df.columns:
        df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")

    return df

def clean_matches_df(matches_raw):
    df = pd.DataFrame(matches_raw).copy()
    if df.empty:
        return df

    rename_map = {
        "id": "match_id",
        "h_team": "home_team",
        "a_team": "away_team",
    }
    df = df.rename(columns=rename_map)

    for col in MATCH_NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "match_id" in df.columns:
        df["match_id"] = pd.to_numeric(df["match_id"], errors="coerce").astype("Int64")
    if "season" in df.columns:
        df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")

    return df

# -----------------------------
# ROSTER / FILTERING HELPERS
# -----------------------------
def parse_player_selection(selection_text):
    if not selection_text or "| id=" not in selection_text:
        return None
    try:
        return int(selection_text.split("| id=")[-1].strip())
    except Exception:
        return None

def split_team_string(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(",") if t.strip()]

def build_team_list_from_roster(roster_df):
    teams = set()
    if roster_df is None or roster_df.empty or "team_title" not in roster_df.columns:
        return []
    for val in roster_df["team_title"].dropna():
        for t in split_team_string(val):
            teams.add(t)
    return sorted(teams)

def filter_roster_for_team(roster_df, team_name):
    if roster_df is None or roster_df.empty:
        return pd.DataFrame()
    if team_name == "All teams":
        return roster_df.copy()

    mask = roster_df["team_title"].apply(lambda x: team_name in split_team_string(x))
    return roster_df.loc[mask].copy()

def build_player_options(roster_df):
    if roster_df is None or roster_df.empty:
        return []

    out = roster_df.copy()
    out = out.dropna(subset=["id", "player_name"])
    out["id"] = out["id"].astype(int)

    label_team = out["team_title"] if "team_title" in out.columns else ""
    label_season = out["selected_season"] if "selected_season" in out.columns else ""

    out["label"] = (
        out["player_name"].astype(str)
        + " | " + label_team.astype(str)
        + " | season=" + label_season.astype(str)
        + " | id=" + out["id"].astype(str)
    )

    out = out.sort_values(["player_name", "team_title", "selected_season"])
    return out["label"].drop_duplicates().tolist()

def filter_matches_by_selection(matches_df, season_choice, team_choice):
    if matches_df is None or matches_df.empty:
        return pd.DataFrame()

    out = matches_df.copy()

    if season_choice != "all" and "season" in out.columns:
        out = out[out["season"] == int(season_choice)]

    if team_choice != "All teams":
        mask = pd.Series(False, index=out.index)
        if "home_team" in out.columns:
            mask = mask | (out["home_team"] == team_choice)
        if "away_team" in out.columns:
            mask = mask | (out["away_team"] == team_choice)
        out = out[mask]

    return out.reset_index(drop=True)

def filter_shots_by_matches(shots_df, valid_match_ids, season_choice=None):
    if shots_df is None or shots_df.empty:
        return pd.DataFrame()

    out = shots_df.copy()

    if "match_id" in out.columns:
        out = out[out["match_id"].isin(valid_match_ids)]

    if season_choice != "all" and "season" in out.columns:
        out = out[out["season"] == int(season_choice)]

    return out.reset_index(drop=True)

def derive_home_away_dummies(df, team_choice):
    out = df.copy()

    out["is_home"] = np.nan
    out["is_away"] = np.nan

    if team_choice != "All teams":
        if "home_team" in out.columns and "away_team" in out.columns:
            out["is_home"] = np.where(out["home_team"] == team_choice, 1, 0)
            out["is_away"] = np.where(out["away_team"] == team_choice, 1, 0)
    else:
        side_col = None
        for c in ["h_a", "side", "match_h_a", "match_side"]:
            if c in out.columns:
                side_col = c
                break
        if side_col is not None:
            out["is_home"] = np.where(out[side_col].astype(str).str.lower().isin(["h", "home"]), 1, 0)
            out["is_away"] = np.where(out[side_col].astype(str).str.lower().isin(["a", "away"]), 1, 0)

    return out

def build_shot_level_table(shots_df, matches_df, player_name, league_code, season_choice, team_choice):
    season_matches = filter_matches_by_selection(matches_df, season_choice, team_choice)

    if season_matches.empty or "match_id" not in season_matches.columns:
        return pd.DataFrame(), pd.DataFrame()

    valid_match_ids = set(season_matches["match_id"].dropna().astype(int).tolist())
    selected_shots = filter_shots_by_matches(shots_df, valid_match_ids, season_choice=season_choice)

    if selected_shots.empty:
        return pd.DataFrame(), season_matches

    match_cols = [
        c for c in [
            "match_id", "date", "season", "home_team", "away_team",
            "time", "goals", "shots", "xG", "xA", "assists", "key_passes",
            "npg", "npxG", "xGChain", "xGBuildup", "position",
            "h_goals", "a_goals", "h_a", "side"
        ] if c in season_matches.columns
    ]

    match_small = season_matches[match_cols].copy()

    rename_map = {
        "date": "match_date",
        "season": "match_season",
        "home_team": "match_home_team",
        "away_team": "match_away_team",
        "time": "match_minutes",
        "goals": "match_goals",
        "shots": "match_shots",
        "xG": "match_xG",
        "xA": "match_xA",
        "assists": "match_assists",
        "key_passes": "match_key_passes",
        "npg": "match_npg",
        "npxG": "match_npxG",
        "xGChain": "match_xGChain",
        "xGBuildup": "match_xGBuildup",
        "position": "match_position",
        "h_goals": "match_home_goals",
        "a_goals": "match_away_goals",
        "h_a": "match_h_a",
        "side": "match_side",
    }
    match_small = match_small.rename(columns=rename_map)

    selected_shots = selected_shots.merge(match_small, on="match_id", how="left")

    if "home_team" not in selected_shots.columns and "match_home_team" in selected_shots.columns:
        selected_shots["home_team"] = selected_shots["match_home_team"]
    if "away_team" not in selected_shots.columns and "match_away_team" in selected_shots.columns:
        selected_shots["away_team"] = selected_shots["match_away_team"]

    selected_shots = derive_home_away_dummies(selected_shots, team_choice)

    if "result" in selected_shots.columns:
        selected_shots["is_goal"] = np.where(selected_shots["result"].astype(str) == "Goal", 1, 0)
    else:
        selected_shots["is_goal"] = np.nan

    first_cols = [
        "player_id", "match_id", "shot_id", "match_date", "match_season",
        "home_team", "away_team", "is_home", "is_away",
        "minute", "result", "is_goal", "situation", "shotType", "X", "Y", "xG",
        "player_assisted", "lastAction",
        "match_minutes", "match_goals", "match_shots", "match_xG", "match_xA",
        "match_assists", "match_key_passes", "match_npg", "match_npxG",
        "match_xGChain", "match_xGBuildup", "match_position",
        "match_home_goals", "match_away_goals"
    ]
    first_cols = [c for c in first_cols if c in selected_shots.columns]
    remaining = [c for c in selected_shots.columns if c not in first_cols]
    selected_shots = selected_shots[first_cols + remaining]

    return selected_shots, season_matches

def build_match_level_table(matches_df, player_name, league_code, season_choice, team_choice):
    out = filter_matches_by_selection(matches_df, season_choice, team_choice)

    if out.empty:
        return out

    out = derive_home_away_dummies(out, team_choice)

    rename_map = {
        "date": "match_date",
        "season": "match_season",
        "time": "match_minutes",
        "goals": "match_goals",
        "shots": "match_shots",
        "xG": "match_xG",
        "xA": "match_xA",
        "assists": "match_assists",
        "key_passes": "match_key_passes",
        "npg": "match_npg",
        "npxG": "match_npxG",
        "xGChain": "match_xGChain",
        "xGBuildup": "match_xGBuildup",
        "position": "match_position",
        "h_goals": "match_home_goals",
        "a_goals": "match_away_goals",
    }
    out = out.rename(columns=rename_map)

    first_cols = [
        "match_id", "match_date", "match_season", "home_team", "away_team",
        "is_home", "is_away", "match_minutes", "match_goals", "match_shots",
        "match_xG", "match_xA", "match_assists", "match_key_passes",
        "match_npg", "match_npxG", "match_xGChain", "match_xGBuildup",
        "match_position", "match_home_goals", "match_away_goals"
    ]
    first_cols = [c for c in first_cols if c in out.columns]
    remaining = [c for c in out.columns if c not in first_cols]
    out = out[first_cols + remaining]

    return out.reset_index(drop=True)

# -----------------------------
# SUMMARY + PLOTS
# -----------------------------
def fmt_num(x, digits=6):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

def shot_summary_html(shots_df):
    if shots_df is None or shots_df.empty:
        return "<b>No shot-level data found for this selection.</b>"

    xg = pd.to_numeric(shots_df["xG"], errors="coerce") if "xG" in shots_df.columns else pd.Series(dtype=float)
    n_shots = len(shots_df)
    n_matches = shots_df["match_id"].nunique() if "match_id" in shots_df.columns else np.nan
    n_goals = int(shots_df["is_goal"].sum()) if "is_goal" in shots_df.columns else np.nan

    shots_per_match = n_shots / n_matches if (pd.notna(n_matches) and n_matches > 0) else np.nan
    goal_rate = n_goals / n_shots if n_shots > 0 and pd.notna(n_goals) else np.nan
    home_share = shots_df["is_home"].mean() if "is_home" in shots_df.columns else np.nan
    away_share = shots_df["is_away"].mean() if "is_away" in shots_df.columns else np.nan

    xg_nonmissing = xg.dropna()
    xg_sd = xg_nonmissing.std(ddof=1) if len(xg_nonmissing) > 1 else 0.0
    xg_var = xg_nonmissing.var(ddof=1) if len(xg_nonmissing) > 1 else 0.0

    return f"""
    <div style="padding:12px 16px; border:1px solid #d9d9d9; border-radius:12px; margin:8px 0 14px 0; background:#fafafa;">
      <div style="font-size:18px; font-weight:700; margin-bottom:8px;">Shot-level summary</div>
      <div style="display:flex; gap:12px; flex-wrap:wrap;">
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Total shots</b><br>{n_shots}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Total goals</b><br>{n_goals}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Total xG</b><br>{fmt_num(xg.sum())}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Mean shot xG</b><br>{fmt_num(xg.mean())}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>SD shot xG</b><br>{fmt_num(xg_sd)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Variance shot xG</b><br>{fmt_num(xg_var)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Shots per match</b><br>{fmt_num(shots_per_match)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Goal rate</b><br>{fmt_num(goal_rate)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Home-shot share</b><br>{fmt_num(home_share)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Away-shot share</b><br>{fmt_num(away_share)}</div>
      </div>
    </div>
    """

def match_summary_html(match_df):
    if match_df is None or match_df.empty:
        return "<b>No match-level data found for this selection.</b>"

    xg = pd.to_numeric(match_df["match_xG"], errors="coerce") if "match_xG" in match_df.columns else pd.Series(dtype=float)
    xa = pd.to_numeric(match_df["match_xA"], errors="coerce") if "match_xA" in match_df.columns else pd.Series(dtype=float)

    xg_nonmissing = xg.dropna()
    xg_sd = xg_nonmissing.std(ddof=1) if len(xg_nonmissing) > 1 else 0.0
    xg_var = xg_nonmissing.var(ddof=1) if len(xg_nonmissing) > 1 else 0.0

    return f"""
    <div style="padding:12px 16px; border:1px solid #d9d9d9; border-radius:12px; margin:8px 0 14px 0; background:#fafafa;">
      <div style="font-size:18px; font-weight:700; margin-bottom:8px;">Match-level summary</div>
      <div style="display:flex; gap:12px; flex-wrap:wrap;">
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Matches</b><br>{len(match_df)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Total match xG</b><br>{fmt_num(xg.sum())}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Mean match xG</b><br>{fmt_num(xg.mean())}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>SD match xG</b><br>{fmt_num(xg_sd)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Variance match xG</b><br>{fmt_num(xg_var)}</div>
        <div style="padding:10px 14px; border-radius:10px; background:white; border:1px solid #ececec;"><b>Mean match xA</b><br>{fmt_num(xa.mean())}</div>
      </div>
    </div>
    """

def plot_shot_outputs(shots_df, plot_out):
    with plot_out:
        plot_out.clear_output(wait=True)

        if shots_df is None or shots_df.empty:
            print("No shot-level data to plot.")
            return

        if "xG" in shots_df.columns:
            vals = pd.to_numeric(shots_df["xG"], errors="coerce").dropna().values
            if len(vals) > 0:
                plt.figure(figsize=(7, 4))
                plt.hist(vals, bins=min(20, max(5, len(vals))))
                plt.title("Shot xG distribution")
                plt.xlabel("Shot xG")
                plt.ylabel("Count")
                plt.show()

        if "result" in shots_df.columns:
            counts = shots_df["result"].astype(str).value_counts()
            if len(counts) > 0:
                plt.figure(figsize=(7, 4))
                counts.plot(kind="bar")
                plt.title("Shot results")
                plt.xlabel("Result")
                plt.ylabel("Count")
                plt.xticks(rotation=45, ha="right")
                plt.show()

        if "situation" in shots_df.columns:
            counts = shots_df["situation"].astype(str).value_counts()
            if len(counts) > 0:
                plt.figure(figsize=(7, 4))
                counts.plot(kind="bar")
                plt.title("Shot situations")
                plt.xlabel("Situation")
                plt.ylabel("Count")
                plt.xticks(rotation=45, ha="right")
                plt.show()

def plot_match_outputs(match_df, plot_out):
    with plot_out:
        plot_out.clear_output(wait=True)

        if match_df is None or match_df.empty:
            print("No match-level data to plot.")
            return

        if "match_xG" in match_df.columns:
            vals = pd.to_numeric(match_df["match_xG"], errors="coerce").dropna().values
            if len(vals) > 0:
                plt.figure(figsize=(7, 4))
                plt.hist(vals, bins=min(15, max(5, len(vals))))
                plt.title("Match xG distribution")
                plt.xlabel("Match xG")
                plt.ylabel("Count")
                plt.show()

# -----------------------------
# UI WIDGETS
# -----------------------------
mode_toggle = widgets.ToggleButtons(
    options=[("Player shot data", "shot"), ("Player match data", "match")],
    value="shot",
    description="Data type:",
    style={"description_width": "initial"}
)

league_dropdown = widgets.Dropdown(
    options=LEAGUE_OPTIONS,
    value="epl",
    description="League:",
    layout=widgets.Layout(width="220px")
)

season_dropdown = widgets.Dropdown(
    options=SEASON_OPTIONS,
    value=2024,
    description="Season:",
    layout=widgets.Layout(width="220px")
)

team_dropdown = widgets.Dropdown(
    options=["All teams"],
    value="All teams",
    description="Team:",
    layout=widgets.Layout(width="260px")
)

player_combobox = widgets.Combobox(
    placeholder="Load league-season, then start typing player name",
    options=[],
    ensure_option=True,
    description="Player:",
    layout=widgets.Layout(width="600px")
)

load_roster_button = widgets.Button(
    description="Load league-season",
    button_style="primary",
    layout=widgets.Layout(width="180px")
)

load_player_button = widgets.Button(
    description="Load player data",
    button_style="success",
    layout=widgets.Layout(width="160px")
)

download_button = widgets.Button(
    description="Download CSV",
    button_style="info",
    layout=widgets.Layout(width="140px")
)

status_out = widgets.Output()
summary_out = widgets.Output()
plot_out = widgets.Output()
table_out = widgets.Output()

# -----------------------------
# UI HELPERS
# -----------------------------
def set_status(msg):
    with status_out:
        status_out.clear_output(wait=True)
        print(msg)

def refresh_team_and_player_widgets():
    roster_df = APP["roster_df"]

    if roster_df is None or roster_df.empty:
        team_dropdown.options = ["All teams"]
        team_dropdown.value = "All teams"
        player_combobox.options = []
        player_combobox.value = ""
        return

    teams = build_team_list_from_roster(roster_df)
    team_dropdown.options = ["All teams"] + teams
    team_dropdown.value = "All teams"

    player_opts = build_player_options(roster_df)
    player_combobox.options = player_opts
    player_combobox.value = ""

# -----------------------------
# CALLBACKS
# -----------------------------
def on_load_roster_clicked(_):
    league_code = league_dropdown.value
    season_choice = season_dropdown.value

    try:
        set_status(f"Loading roster for {league_code}, season={season_choice} ...")

        if season_choice == "all":
            roster_parts = []

            for yr in SEASON_START_YEARS:
                try:
                    players_raw, teams_raw = run_async(fetch_league_players_and_teams_one(league_code, yr))
                    players_df = clean_players_df(players_raw, league_code, yr)
                    if not players_df.empty:
                        roster_parts.append(players_df)
                except Exception:
                    pass

            if roster_parts:
                roster_df = pd.concat(roster_parts, ignore_index=True).drop_duplicates()
            else:
                roster_df = pd.DataFrame()

        else:
            players_raw, teams_raw = run_async(fetch_league_players_and_teams_one(league_code, int(season_choice)))
            roster_df = clean_players_df(players_raw, league_code, int(season_choice))

        APP["roster_df"] = roster_df
        APP["current_table"] = None
        APP["current_mode"] = None
        APP["current_league"] = league_code
        APP["current_season"] = season_choice
        APP["current_team"] = None
        APP["current_player_name"] = None

        refresh_team_and_player_widgets()

        with summary_out:
            summary_out.clear_output(wait=True)
        with plot_out:
            plot_out.clear_output(wait=True)
        with table_out:
            table_out.clear_output(wait=True)

        set_status(f"Roster loaded. Rows: {len(roster_df):,}. Now choose team and player.")

    except Exception as e:
        set_status(f"Error loading roster: {e}")

def on_team_change(change):
    if change["name"] != "value":
        return

    roster_df = APP["roster_df"]
    team_choice = team_dropdown.value

    roster_filtered = filter_roster_for_team(roster_df, team_choice)
    player_opts = build_player_options(roster_filtered)
    player_combobox.options = player_opts
    player_combobox.value = ""

def on_load_player_clicked(_):
    selection_text = player_combobox.value.strip()
    player_id = parse_player_selection(selection_text)

    if player_id is None:
        set_status("Choose a player from the dropdown suggestions so the player ID is captured.")
        return

    league_code = league_dropdown.value
    season_choice = season_dropdown.value
    team_choice = team_dropdown.value
    mode = mode_toggle.value

    roster_df = APP["roster_df"]
    if roster_df is None or roster_df.empty:
        set_status("Load a league-season first.")
        return

    row = roster_df.loc[roster_df["id"] == player_id]
    if row.empty:
        set_status("Selected player not found in the loaded roster.")
        return

    player_name = row.iloc[0]["player_name"]

    try:
        set_status(f"Loading player data for {player_name} ...")
        shots_raw, matches_raw, grouped_raw = run_async(fetch_player_bundle(player_id))

        shots_df = clean_shots_df(shots_raw)
        matches_df = clean_matches_df(matches_raw)

        if mode == "shot":
            current_table, filtered_matches = build_shot_level_table(
                shots_df=shots_df,
                matches_df=matches_df,
                player_name=player_name,
                league_code=league_code,
                season_choice=season_choice,
                team_choice=team_choice
            )

            APP["current_table"] = current_table
            APP["current_mode"] = "shot"
            APP["current_player_name"] = player_name
            APP["current_team"] = team_choice

            with summary_out:
                summary_out.clear_output(wait=True)
                display(widgets.HTML(shot_summary_html(current_table)))

            plot_shot_outputs(current_table, plot_out)

            with table_out:
                table_out.clear_output(wait=True)
                if current_table.empty:
                    print("No shot rows found for this selection.")
                else:
                    print(f"Rows: {len(current_table):,}")
                    display(current_table.style.set_table_attributes('style="display:block; overflow-x:auto; white-space:nowrap;"'))

            set_status("Shot-level data loaded." if not current_table.empty else "No shot rows found for this selection.")

        else:
            current_table = build_match_level_table(
                matches_df=matches_df,
                player_name=player_name,
                league_code=league_code,
                season_choice=season_choice,
                team_choice=team_choice
            )

            APP["current_table"] = current_table
            APP["current_mode"] = "match"
            APP["current_player_name"] = player_name
            APP["current_team"] = team_choice

            with summary_out:
                summary_out.clear_output(wait=True)
                display(widgets.HTML(match_summary_html(current_table)))

            plot_match_outputs(current_table, plot_out)

            with table_out:
                table_out.clear_output(wait=True)
                if current_table.empty:
                    print("No match rows found for this selection.")
                else:
                    print(f"Rows: {len(current_table):,}")
                    display(current_table.style.set_table_attributes('style="display:block; overflow-x:auto; white-space:nowrap;"'))

            set_status("Match-level data loaded." if not current_table.empty else "No match rows found for this selection.")

    except Exception as e:
        set_status(f"Error loading player data: {e}")

def on_download_clicked(_):
    df = APP["current_table"]

    if df is None or df.empty:
        set_status("Nothing to download yet. Load a player first.")
        return

    player_name = (APP["current_player_name"] or "player").replace(" ", "_")
    league_code = APP["current_league"] or "league"
    season_choice = APP["current_season"]
    season_txt = str(season_choice)
    mode = APP["current_mode"] or "data"

    filename = f"{player_name}_{league_code}_{season_txt}_{mode}.csv"
    df.to_csv(filename, index=False)

    if IN_COLAB and colab_files is not None:
        colab_files.download(filename)
        set_status(f"CSV prepared: {filename}")
    else:
        set_status(f"CSV saved as {filename}")

# bind
load_roster_button.on_click(on_load_roster_clicked)
team_dropdown.observe(on_team_change, names="value")
load_player_button.on_click(on_load_player_clicked)
download_button.on_click(on_download_clicked)

# -----------------------------
# LAYOUT
# -----------------------------
header = widgets.HTML("""
<h2 style="margin-bottom:6px;">Understat Player Explorer</h2>
<p style="margin-top:0; color:#444;">
Choose a data type, league, season, team, and player. Shot mode returns shot-level rows with match context merged in.
</p>
""")

row0 = widgets.HBox([mode_toggle])
row1 = widgets.HBox([league_dropdown, season_dropdown, load_roster_button], layout=widgets.Layout(gap="10px"))
row2 = widgets.HBox([team_dropdown, player_combobox], layout=widgets.Layout(gap="10px"))
row3 = widgets.HBox([load_player_button, download_button], layout=widgets.Layout(gap="10px"))

ui = widgets.VBox([
    header,
    row0,
    row1,
    row2,
    row3,
    status_out,
    summary_out,
    plot_out,
    table_out
])

display(ui)
set_status("Explorer loaded. Click 'Load league-season'.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>